# Gradient descent variants

Notebook 10 solved linear regression in one step with the normal equation, and also by walking downhill. The walk is the one that scales. It works for models with no closed form, and every neural network in section 05 is trained this way. This notebook builds three versions of it from scratch and shows what each setting does.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from common.datasets import cap_housing_outliers, load_california_housing

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

You are on a hill in thick fog and want the valley floor. You can only feel the slope under your feet. So you step downhill, feel again, and repeat. The size of each step is the learning rate.

The three variants differ in how carefully you feel the slope before each step.

- **Batch:** measure the slope using every data point. Accurate, but each step is slow on big data.
- **Stochastic (SGD):** use one random point. Very cheap and noisy, so the path wobbles.
- **Mini-batch:** use a small random group (32 to 256 rows). A compromise, and the standard in deep learning, because it also keeps GPUs busy.

## 2. Maths

Cost over $n$ rows, with $A$ the feature matrix including a bias column:

$$J(\mathbf w)=\frac1n\sum_{i=1}^n\bigl(y_i-\mathbf a_i^\top\mathbf w\bigr)^2,\qquad
\nabla J=-\frac2n\sum_{i=1}^n\mathbf a_i\bigl(y_i-\mathbf a_i^\top\mathbf w\bigr)$$

Every variant takes the same step, $\mathbf w\leftarrow\mathbf w-\eta\,\mathbf g$. They differ in how $\mathbf g$ is estimated:

- Batch: $\mathbf g$ is the full sum above.
- SGD: the same formula with a single random row.
- Mini-batch: the same formula averaged over $B$ random rows.

The one-row and mini-batch gradients are noisy but correct on average, so the walk still goes downhill in expectation. The noise shrinks as $B$ grows, roughly like $1/\sqrt B$.

**Learning rate.** For this quadratic cost, batch descent converges only if $\eta<2/\lambda_{\max}$, where $\lambda_{\max}$ is the largest eigenvalue of the Hessian $\tfrac2nA^\top A$. Too high and each step overshoots the valley further than the last.

**Momentum** keeps a running velocity so steps build up along a consistent direction and cancel across a zigzag:

$$\mathbf v\leftarrow\beta\mathbf v+\mathbf g,\qquad \mathbf w\leftarrow\mathbf w-\eta\,\mathbf v$$

With $\beta=0.9$ the velocity is roughly an average of the last 10 gradients.

## 3. Data

Capped and standardised housing features (notebook 10 showed why), 2,000 rows, with a bias column. I compute the exact answer first so we know where the walk should end.

In [ ]:
X, y = load_california_housing(n_samples=2000, seed=SEED)
X = cap_housing_outliers(X)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED)
scaler = StandardScaler().fit(X_tr)
A_tr = np.column_stack([np.ones(len(X_tr)), scaler.transform(X_tr)])
A_te = np.column_stack([np.ones(len(X_te)), scaler.transform(X_te)])
y_tr_a, y_te_a = y_tr.to_numpy(), y_te.to_numpy()

w_star = np.linalg.solve(A_tr.T @ A_tr, A_tr.T @ y_tr_a)
J_star = float(np.mean((y_tr_a - A_tr @ w_star) ** 2))
lam_max = np.linalg.eigvalsh(2 / len(A_tr) * A_tr.T @ A_tr).max()
print("minimum cost J*:", J_star)
print("largest Hessian eigenvalue:", lam_max, " -> stable only if lr <", 2 / lam_max)

## 4. From scratch

One function covers all three variants. `batch_size=None` is batch descent, `1` is SGD, anything between is mini-batch. It records the full-data cost after every epoch (one pass over the data), plus the cost after each of the first few updates so we can see the noise.

In [ ]:
def gradient_descent(
    A: np.ndarray,
    target: np.ndarray,
    lr: float,
    epochs: int,
    batch_size: int | None = None,
    momentum: float = 0.0,
    decay: float = 0.0,
    seed: int = 0,
    trace: int = 0,
) -> tuple[np.ndarray, list[float], list[float]]:
    n = len(target)
    size = n if batch_size is None else batch_size
    order_rng = np.random.default_rng(seed)
    w, v = np.zeros(A.shape[1]), np.zeros(A.shape[1])
    epoch_cost, update_cost, t = [], [], 0
    with np.errstate(all="ignore"):  # a diverging run should record inf, not warn
        for _ in range(epochs):
            epoch_cost.append(float(np.mean((target - A @ w) ** 2)))
            order = order_rng.permutation(n)
            for start in range(0, n, size):
                idx = order[start : start + size]
                g = -2 / len(idx) * A[idx].T @ (target[idx] - A[idx] @ w)
                v = momentum * v + g
                w = w - lr / (1 + decay * t) * v
                t += 1
                if t <= trace:
                    update_cost.append(float(np.mean((target - A @ w) ** 2)))
        epoch_cost.append(float(np.mean((target - A @ w) ** 2)))
    return w, epoch_cost, update_cost

## 5. Batch vs SGD vs mini-batch

Same learning rate for batch and mini-batch. SGD needs a smaller one, because each single-row gradient is noisy. Plot the gap to the minimum on a log scale, per epoch and per update.

In [ ]:
runs = {
    "batch": {"lr": 0.3, "batch_size": None},
    "mini-batch (32)": {"lr": 0.05, "batch_size": 32},
    "SGD (1 row)": {"lr": 0.005, "batch_size": 1},
}
results = {}
for name, kw in runs.items():
    results[name] = gradient_descent(A_tr, y_tr_a, epochs=30, trace=150, **kw)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
for name, (w, ec, uc) in results.items():
    a1.plot(np.array(ec) - J_star + 1e-6, label=name)
    a2.plot(uc, label=name)
a1.set_yscale("log")
a1.set_xlabel("epoch")
a1.set_ylabel("cost minus minimum")
a1.legend()
a2.set_xlabel("update (first 150)")
a2.set_ylabel("training cost")
a2.set_ylim(0.4, 2)
plt.show()
for name, (w, ec, uc) in results.items():
    print(
        f"{name:>16}: final cost {ec[-1]:.4f} (min {J_star:.4f}), max weight gap {np.abs(w - w_star).max():.3f}"
    )

Left, per epoch: all three head for the minimum. Mini-batch and SGD get close in a handful of epochs, because they make 63 and 1,500 updates per epoch against 1 for batch. Batch is smooth but slow here. Its cost is within 2% of the minimum while a few weights are still some way off. The cost is nearly flat along those directions, so they barely matter for the fit.

Right, per update: batch moves smoothly but each step costs a full pass. SGD and mini-batch are jagged but cheap, and they make progress from the very first update. For data with millions of rows, you do not wait for a full pass to take a step. That is why they win at scale.

The jaggedness never fully goes away with a fixed learning rate. The weights keep jittering around the minimum, and the jitter is larger when the learning rate or the batch noise is larger.

## 6. Tuning the learning rate

The learning rate is the main setting. Try five values on batch descent. The stability limit computed above sets the edge.

In [ ]:
limit = 2 / lam_max
plt.figure()
for lr in (0.001, 0.02, 0.2, 0.9 * limit, 1.05 * limit):
    _, ec, _ = gradient_descent(A_tr, y_tr_a, lr=lr, epochs=40)
    plt.plot(np.clip(ec, 0, 10), label=f"lr={lr:.3f}")
    print(f"lr={lr:.3f}: cost after 5 epochs {ec[5]:.3f}, after 40 epochs {ec[-1]:.3g}")
plt.axhline(J_star, ls="--", c="grey")
plt.ylim(0, 6)
plt.xlabel("epoch")
plt.ylabel("training cost")
plt.legend()
plt.show()
print("stability limit:", limit)

- **Too small (0.001):** smooth and safe, but the cost has barely moved after 40 epochs.
- **Small (0.02):** heading the right way, still well short of the minimum after 40 epochs.
- **Moderate (0.2):** drops fast and flattens.
- **Just under the limit (0.9 times it):** the fastest here, but the path bounces from side to side along the steepest direction while it shrinks. There is little safety margin.
- **Above the limit:** each step lands further up the other wall. The cost climbs without bound.

Reading the curve is the diagnostic. A falling, flattening curve is healthy. A flat one means the rate is too small or you are done. A cost that rises or jumps about means the rate is too high.

A common middle path is to start big and shrink the rate. With SGD this also stops the endless jitter. Compare a constant rate and a decaying one, $\eta_t=\eta_0/(1+\text{decay}\cdot t)$.

In [ ]:
fig, ax = plt.subplots()
for name, decay in [("constant", 0.0), ("decay", 0.01)]:
    _, ec, _ = gradient_descent(A_tr, y_tr_a, lr=0.01, epochs=30, batch_size=1, decay=decay)
    ax.plot(np.array(ec) - J_star + 1e-6, label=name)
    print(f"{name:>8}: final cost gap {ec[-1] - J_star:.5f}")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("cost minus minimum")
ax.legend()
plt.show()

## 7. Momentum, briefly

Momentum builds speed along a steady slope. Count how many epochs batch descent needs to get within 1% of the minimum cost with and without it.

In [ ]:
def epochs_to_close(ec: list[float], tol: float = 0.01) -> int | None:
    hit = np.where(np.array(ec) <= J_star * (1 + tol))[0]
    return int(hit[0]) if len(hit) else None


for lr, beta in [(0.02, 0.0), (0.02, 0.9), (0.05, 0.9)]:
    _, ec, _ = gradient_descent(A_tr, y_tr_a, lr=lr, epochs=300, momentum=beta)
    print(f"lr={lr}, momentum={beta}: epochs to within 1% -> {epochs_to_close(ec)}")

At the same learning rate, momentum arrives in 69 epochs and plain descent has not by epoch 300. Raise the rate with momentum on and it can overshoot, so the two settings need tuning together. Adam and similar optimisers in section 05 are momentum plus a per-weight learning rate.

## 8. The library version

`SGDRegressor` does mini-batch-style SGD with a decaying rate and optional early stopping. Scale first, always. Here it sits in a pipeline and is compared with the exact solution on held-out data.

In [ ]:
exact_pred = A_te @ w_star
sgd = make_pipeline(
    StandardScaler(),
    SGDRegressor(eta0=0.01, max_iter=2000, tol=1e-6, random_state=SEED),
).fit(X_tr, y_tr)
print("exact test MSE:", mean_squared_error(y_te, exact_pred))
print("SGDRegressor  :", mean_squared_error(y_te, sgd.predict(X_te)))
print("epochs run    :", sgd[-1].n_iter_)

Test error is within 1% of the exact answer. Its main knobs are `eta0` (starting rate), `learning_rate` (the decay schedule), `alpha` (a penalty, see notebook 13) and `early_stopping`.

## 9. Early stopping as regularisation

Start the weights at zero and walk. The first steps learn the big, reliable patterns and later steps start fitting noise. So stopping early limits how far the weights travel, which acts like a penalty. To see it, use few rows and many features: 100 training rows with degree-2 features (44 columns).

In [ ]:
X_s, y_s = load_california_housing(n_samples=800, seed=SEED)
X_s = cap_housing_outliers(X_s)
Xs_tr, Xs_va, ys_tr, ys_va = train_test_split(X_s, y_s, test_size=700, random_state=SEED)
prep = make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler()).fit(Xs_tr)
P_tr = np.column_stack([np.ones(len(Xs_tr)), prep.transform(Xs_tr)])
P_va = np.column_stack([np.ones(len(Xs_va)), prep.transform(Xs_va)])

w, tr_curve, va_curve = np.zeros(P_tr.shape[1]), [], []
for epoch in range(3000):
    w = w + 0.02 * (2 / len(ys_tr)) * P_tr.T @ (ys_tr.to_numpy() - P_tr @ w)
    tr_curve.append(np.mean((ys_tr.to_numpy() - P_tr @ w) ** 2))
    va_curve.append(np.mean((ys_va.to_numpy() - P_va @ w) ** 2))

best = int(np.argmin(va_curve))
plt.plot(tr_curve, label="train")
plt.plot(va_curve, label="validation")
plt.axvline(best, ls="--", c="grey")
plt.ylim(0.2, 1.2)
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()
print("best epoch:", best, " val MSE:", va_curve[best], " at epoch 3000:", va_curve[-1])

Training error keeps falling, but validation error bottoms out and creeps up. Stop at the minimum. `SGDRegressor(early_stopping=True, n_iter_no_change=...)` does this automatically with a patience counter, which matters when the validation curve is noisy.

## 10. What breaks

- **Learning rate too high.** The cost rises or becomes `inf`. Lower it by a factor of 3 to 10.
- **Unscaled features.** One rate cannot suit every direction of the valley. Standardise.
- **Constant rate with SGD.** The weights never settle. Decay the rate, or average the last iterates.
- **Non-convex costs.** Linear regression has one bowl. Neural networks have many dips and flat patches, so the start point and the rate matter far more.
- **Stopping too early.** A flat cost curve at a high value is not convergence. Compare with a known minimum where you can.

## Summary

- Batch, SGD and mini-batch differ only in how many rows estimate each gradient. Mini-batch wins on cost per update and smoothness.
- The learning rate must sit below the stability limit and well above tiny. Read the cost curve to set it.
- Decay the rate to quieten SGD. Momentum speeds up consistent directions.
- Stopping early limits weight growth and works as regularisation.

Next: `13_bias_variance_regularisation.ipynb`.